# Model Revenue Prediction

This notebook runs the time-aware feature engineering and model suite for Datathon 2026 Part 3. The target grain is one row per `Date`; we forecast `Revenue` and `COGS` for the submission horizon. Metrics follow the PDF: `MAE`, `RMSE`, and `R2`, with `MAE` treated as the primary model-selection criterion.

Leakage policy: same-day future transaction/web/inventory/order signals are not used directly. The feature store converts them into historical lag or shifted rolling features. Target lags are handled recursively during the ML forecast.

Important Kaggle note: public leaderboard feedback showed that pure recursive ML and local robust blend underperform sample-based anchors. Therefore the final `submission_model_revenue_prediction.csv` defaults to `lb_calibrated_sample`, using documented public-LB scale factors around the sample shape. The train-only candidates are still saved separately for diagnostics.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
from IPython.display import Image, Markdown, display
import pandas as pd

from src.models import ForecastConfig, run_revenue_prediction_pipeline, summarize_probe_scores

## Configuration

`RUN_H2O_AUTOML=True` runs H2O AutoML as a reference benchmark. It can take extra time and requires Java/H2O to work in the local environment. If the environment is unstable, set it to `False`; the core sklearn/XGBoost pipeline will still run. Fine tuning is enabled by default and uses the latest temporal validation fold to choose a small MAE-focused tree-model grid.

In [ ]:
RUN_H2O_AUTOML = True

config = ForecastConfig(
    data_dir=PROJECT_ROOT / 'data',
    output_dir=PROJECT_ROOT / 'outputs/model_revenue_prediction',
    validation_years=(2020, 2021, 2022),
    run_h2o=RUN_H2O_AUTOML,
    h2o_max_runtime_secs=90,
    h2o_max_models=12,
    enable_fine_tuning=True,
    fine_tune_validation_year=2022,
    tune_model_families=('hist_gradient_boosting',),
    final_submission_strategy='lb_calibrated_sample',
    save_outputs=True,
)
config

## Run Feature Engineering and Model Suite

The pipeline trains Linear Regression, Ridge, Lasso, HistGradientBoosting, RandomForest, XGBoost, and LightGBM if installed. LightGBM is optional: if the package is not available, it is skipped without breaking the run.

It also builds a robust horizon-safe blend from seasonal/Q-specialist models. This blend is tuned on rolling 548-day backtests, which better matches the Kaggle test length than one-step yearly CV.

In [ ]:
result = run_revenue_prediction_pipeline(config)

display(Markdown(f"Outputs saved to `{result['output_dir']}`"))
display(Markdown(f"Best ML models by mean MAE: `{result['best_model_names']}`"))
display(Markdown(f"Final submission strategy: `{config.final_submission_strategy}`"))

## Public-LB Probe Suite

The current best public result comes from a calibrated `sample_submission` anchor. To push beyond it, the pipeline generates orthogonal candidate submissions that each test one hypothesis: overall level, target imbalance, year imbalance, daily volatility, seasonal peak/trough shape, or 2024 within-year trend. Submit them in priority order and feed the public-LB scores back into the next calibration pass.

In [ ]:
probe_manifest = result['lb_calibrated_sample'].get('probe_manifest')
if probe_manifest is not None and not probe_manifest.empty:
    display(probe_manifest[['priority', 'file', 'description', 'mean_abs_delta_vs_lbcal', 'revenue_mean', 'cogs_mean']].style.format({
        'mean_abs_delta_vs_lbcal': '{:,.0f}',
        'revenue_mean': '{:,.0f}',
        'cogs_mean': '{:,.0f}',
    }))
else:
    display(Markdown('No public-LB probe suite was generated.'))

score_template_path = result['output_dir'] / 'submission_probe_score_template.csv'
if score_template_path.exists():
    score_summary = summarize_probe_scores(score_template_path)
    if len(score_summary) > 1:
        display(Markdown('### Submitted Probe Score Summary'))
        display(score_summary[['priority', 'file', 'public_mae', 'delta_vs_anchor', 'improved_anchor']].style.format({
            'public_mae': '{:,.2f}',
            'delta_vs_anchor': '{:+,.2f}',
        }))

## Validation Metrics

The table below reports `MAE`, `RMSE`, and `R2` by target/model, averaged across walk-forward validation years. The `_std` columns are the fold-to-fold standard deviation, so they tell us whether a model is stable over time or only good in one year. Lower `MAE_mean` is the most important criterion.

In [ ]:
summary = result['summary'].copy()
display(summary.style.format({
    'mae_mean': '{:,.0f}', 'mae_std': '{:,.0f}',
    'rmse_mean': '{:,.0f}', 'rmse_std': '{:,.0f}',
    'r2_mean': '{:.4f}', 'r2_std': '{:.4f}',
    'n_features': '{:,.0f}', 'n_folds': '{:,.0f}',
}))

metrics = result['metrics'].copy()
display(metrics.sort_values(['target', 'validation_year', 'mae']).style.format({'mae': '{:,.0f}', 'rmse': '{:,.0f}', 'r2': '{:.4f}'}))

## Feature Importance and SHAP

For tree models, the pipeline exports built-in feature importance when available. For models such as HistGradientBoosting without built-in importance, it uses permutation importance measured by MAE degradation. SHAP summary plots are generated when the fitted best model supports `TreeExplainer`.

In [ ]:
for target, importance in result['feature_importance'].items():
    display(Markdown(f"### {target} Feature Importance"))
    display(importance.head(25).style.format({'importance': '{:,.4f}', 'abs_importance': '{:,.4f}'}))
    image_path = result['output_dir'] / f"feature_importance_{target.lower()}.png"
    if image_path.exists():
        display(Image(filename=str(image_path)))
    shap_path = result['shap_paths'].get(target)
    if shap_path is not None and Path(shap_path).exists():
        display(Markdown(f"### {target} SHAP Summary"))
        display(Image(filename=str(shap_path)))

## H2O AutoML Reference

This is a reference benchmark only. The final production pipeline should still remain reproducible and leakage-controlled.

In [ ]:
if not result['h2o_leaderboard'].empty:
    display(result['h2o_leaderboard'])
else:
    display(Markdown('H2O AutoML was skipped or unavailable.'))

## Robust Horizon Blend

The public leaderboard gap suggests that pure recursive ML overfits one-step validation. This diagnostic shows the long-horizon blend weights and candidate backtest metrics. The robust blend is now treated as a diagnostic candidate, not the default final submission, because public leaderboard feedback was worse than the sample anchor.

In [ ]:
robust = result['robust_blend']
display(Markdown('### Tuned Blend Weights'))
display(robust['weights'].style.format({'weight': '{:.2f}', 'cv_tuned_mae': '{:,.0f}'}))

robust_metrics = robust['metrics'].groupby(['target', 'model'], as_index=False).agg(
    mae_mean=('mae', 'mean'),
    mae_std=('mae', 'std'),
    rmse_mean=('rmse', 'mean'),
    r2_mean=('r2', 'mean'),
)
display(Markdown('### Candidate 548-Day Backtest'))
display(robust_metrics.sort_values(['target', 'mae_mean']).style.format({
    'mae_mean': '{:,.0f}', 'mae_std': '{:,.0f}',
    'rmse_mean': '{:,.0f}', 'r2_mean': '{:.4f}',
} ))

display(Markdown(f"Robust blend saved to `{robust['path']}`"))

## Submission Preview

The generated submission keeps the same columns required by `sample_submission.csv`: `Date`, `Revenue`, and `COGS`.

In [ ]:
submission = result['submission'].copy()
display(submission.head())
display(submission.tail())

submission_path = result['output_dir'] / 'submission_model_revenue_prediction.csv'
display(Markdown(f"Submission saved to `{submission_path}`"))